In [1]:
!pip -q install pymupdf pdfplumber pandas openpyxl tabulate
!pip -q install camelot-py[cv] || true


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import json
import re
import os
from pathlib import Path
from typing import List, Dict, Any, Optional

import fitz  # pymupdf
import pdfplumber
import pandas as pd
from tabulate import tabulate

In [11]:
# ====== CẤU HÌNH ======
PDF_PATH = Path("../data/raw/Ngo-doc.pdf")
OUTPUT_DIR = Path("../data/processed/pdf_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH = OUTPUT_DIR / "processed_chunks.jsonl"
TABLES_XLSX_PATH = OUTPUT_DIR / "extracted_tables.xlsx"
MARKDOWN_DIR = OUTPUT_DIR / "markdown_pages"
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

# Tùy chọn
KEEP_PAGE_TEXT = True
KEEP_TABLES = True
SAVE_PAGE_MARKDOWN = True
SAVE_TABLE_CSV = True

In [4]:
def normalize_text(text: str) -> str:
    """Làm sạch text PDF nhưng giữ cấu trúc tương đối."""
    if not text:
        return ""

    # bỏ ngắt dòng kiểu nối từ
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # đổi xuống dòng đơn lẻ thành khoảng trắng nếu bị ngắt giữa câu
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)

    # gom nhiều khoảng trắng
    text = re.sub(r"[ \t]+", " ", text)

    # gom nhiều dòng trống
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def is_meaningful_text(text: str, min_chars: int = 30) -> bool:
    return bool(text and len(text.strip()) >= min_chars)


def safe_filename(name: str) -> str:
    name = re.sub(r"[^\w\-\.]+", "_", name, flags=re.UNICODE)
    return name.strip("_")


def dataframe_to_markdown(df: pd.DataFrame) -> str:
    df2 = df.copy()
    df2 = df2.fillna("")
    return tabulate(df2, headers="keys", tablefmt="github", showindex=False)


def dataframe_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Dọn bảng thô: bỏ cột/ràng trống, chuẩn hóa cell."""
    if df is None or df.empty:
        return df

    df = df.copy()
    df = df.replace(r"^\s*$", "", regex=True)
    df = df.applymap(lambda x: "" if x is None else str(x).strip())

    # bỏ cột toàn rỗng
    df = df.loc[:, (df != "").any(axis=0)]

    # bỏ dòng toàn rỗng
    df = df.loc[(df != "").any(axis=1)]

    df = df.reset_index(drop=True)
    return df


def looks_like_table(df: pd.DataFrame) -> bool:
    """Heuristic đơn giản để lọc bảng có ích."""
    if df is None or df.empty:
        return False
    if df.shape[0] < 2 or df.shape[1] < 2:
        return False
    non_empty_ratio = (df.astype(str).applymap(lambda x: x.strip() != "").values.sum()) / (df.shape[0] * df.shape[1])
    return non_empty_ratio >= 0.25

In [5]:
def extract_page_text_mupdf(doc: fitz.Document, page_index: int) -> str:
    page = doc[page_index]
    text = page.get_text("text", sort=True)
    text = normalize_text(text)
    return text


def get_page_basic_info(doc: fitz.Document, page_index: int) -> Dict[str, Any]:
    page = doc[page_index]
    text = extract_page_text_mupdf(doc, page_index)
    images = page.get_images(full=True)

    return {
        "page_number": page_index + 1,
        "text_len": len(text),
        "has_text_layer": is_meaningful_text(text, min_chars=30),
        "image_count": len(images),
    }

In [6]:
def extract_tables_pdfplumber(pdf_path: Path, page_number_1based: int) -> List[pd.DataFrame]:
    """
    Trích bảng trên 1 trang.
    Dùng 2 chiến lược: lines và stream.
    page_number_1based: số trang tính từ 1.
    """
    tables_out: List[pd.DataFrame] = []

    with pdfplumber.open(str(pdf_path)) as pdf:
        if page_number_1based < 1 or page_number_1based > len(pdf.pages):
            return tables_out

        page = pdf.pages[page_number_1based - 1]

        settings_lines = {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "edge_min_length": 3,
            "intersection_tolerance": 3,
            "text_tolerance": 2,
        }

        settings_stream = {
            "vertical_strategy": "text",
            "horizontal_strategy": "text",
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "text_tolerance": 2,
        }

        for settings in (settings_lines, settings_stream):
            try:
                raw_tables = page.extract_tables(table_settings=settings) or []
                for raw in raw_tables:
                    if not raw:
                        continue
                    # chuẩn hóa thành dataframe
                    max_cols = max(len(r) for r in raw)
                    normalized = []
                    for r in raw:
                        r = list(r) + [""] * (max_cols - len(r))
                        normalized.append(r)
                    df = pd.DataFrame(normalized)
                    df = dataframe_cleanup(df)
                    if looks_like_table(df):
                        tables_out.append(df)
            except Exception:
                pass

    # loại bảng trùng nhau (rất thô)
    deduped = []
    seen = set()
    for df in tables_out:
        sig = df.astype(str).to_csv(index=False)
        if sig not in seen:
            seen.add(sig)
            deduped.append(df)

    return deduped

In [7]:
def extract_tables_camelot(pdf_path: Path, page_number_1based: int) -> List[pd.DataFrame]:
    """
    Camelot tốt với PDF vector có đường kẻ bảng.
    Nếu máy không có ghostscript hoặc camelot lỗi thì trả về [].
    """
    try:
        import camelot
    except Exception:
        return []

    tables_out: List[pd.DataFrame] = []

    for flavor in ("lattice", "stream"):
        try:
            tables = camelot.read_pdf(
                str(pdf_path),
                pages=str(page_number_1based),
                flavor=flavor,
                strip_text="\n",
            )
            for t in tables:
                df = t.df.copy()
                df = dataframe_cleanup(df)
                if looks_like_table(df):
                    tables_out.append(df)
        except Exception:
            continue

    # dedupe
    deduped = []
    seen = set()
    for df in tables_out:
        sig = df.astype(str).to_csv(index=False)
        if sig not in seen:
            seen.add(sig)
            deduped.append(df)

    return deduped

In [8]:
def extract_page_payload(pdf_path: Path, page_index: int) -> Dict[str, Any]:
    doc = fitz.open(str(pdf_path))
    page = doc[page_index]
    page_number = page_index + 1

    text = extract_page_text_mupdf(doc, page_index)
    info = get_page_basic_info(doc, page_index)

    tables: List[Dict[str, Any]] = []
    if KEEP_TABLES:
        # ưu tiên camelot nếu có, nếu không thì pdfplumber
        camelot_tables = extract_tables_camelot(pdf_path, page_number)
        plumber_tables = extract_tables_pdfplumber(pdf_path, page_number)

        all_dfs = camelot_tables + plumber_tables

        # dedupe cuối cùng
        unique = []
        seen = set()
        for df in all_dfs:
            sig = df.astype(str).to_csv(index=False)
            if sig not in seen:
                seen.add(sig)
                unique.append(df)

        for i, df in enumerate(unique, start=1):
            tables.append({
                "table_index": i,
                "rows": df.shape[0],
                "cols": df.shape[1],
                "markdown": dataframe_to_markdown(df),
                "csv": df.to_csv(index=False),
            })

    # tạo payload trang
    payload = {
        "doc_name": pdf_path.name,
        "page_number": page_number,
        "has_text_layer": info["has_text_layer"],
        "image_count": info["image_count"],
        "text": text if KEEP_PAGE_TEXT else "",
        "tables": tables,
    }
    doc.close()
    return payload

In [9]:
def split_text_into_chunks(text: str, max_chars: int = 1200, overlap: int = 120) -> List[str]:
    """
    Chia text theo đoạn, giữ overlap nhẹ.
    """
    text = normalize_text(text)
    if not text:
        return []

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    for p in paragraphs:
        if not current:
            current = p
            continue

        if len(current) + 2 + len(p) <= max_chars:
            current += "\n\n" + p
        else:
            chunks.append(current)
            if overlap > 0:
                tail = current[-overlap:]
                current = tail + "\n\n" + p
            else:
                current = p

    if current.strip():
        chunks.append(current)

    # lọc trùng thô
    cleaned = []
    seen = set()
    for c in chunks:
        c2 = c.strip()
        if c2 and c2 not in seen:
            seen.add(c2)
            cleaned.append(c2)
    return cleaned


def build_chunks_from_page(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    page_number = payload["page_number"]

    # chunk text
    if payload["text"]:
        for idx, chunk in enumerate(split_text_into_chunks(payload["text"]), start=1):
            chunks.append({
                "doc_name": payload["doc_name"],
                "page_number": page_number,
                "chunk_type": "text",
                "chunk_index": idx,
                "content": chunk,
                "metadata": {
                    "has_text_layer": payload["has_text_layer"],
                    "image_count": payload["image_count"],
                }
            })

    # chunk tables
    for t in payload["tables"]:
        chunks.append({
            "doc_name": payload["doc_name"],
            "page_number": page_number,
            "chunk_type": "table",
            "chunk_index": t["table_index"],
            "content": t["markdown"],
            "metadata": {
                "rows": t["rows"],
                "cols": t["cols"],
            }
        })

    return chunks

In [12]:
def process_pdf(pdf_path: Path) -> List[Dict[str, Any]]:
    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)
    doc.close()

    all_payloads: List[Dict[str, Any]] = []

    for page_index in range(total_pages):
        print(f"Processing page {page_index + 1}/{total_pages} ...")
        payload = extract_page_payload(pdf_path, page_index)
        all_payloads.append(payload)

    return all_payloads


payloads = process_pdf(PDF_PATH)
len(payloads), payloads[0].keys()

Processing page 1/228 ...


Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 90 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 208 0 (offset 0)
Ignoring wrong pointing object 210 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset 0)
Ignoring wrong pointing object 265 0 (offset 0)
Ignoring wrong pointing object 426 0 (offset 0)
Ignoring wrong pointing object 622 0 (offset 0)
Ignoring wrong pointing object 623 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 626 0 (offset 0)
Ignoring wrong pointing object 627 0 (offset 0)
Ignoring wrong pointing object 633 0 (offset 0)
Ignoring wrong pointing object 647 0 (offset 0)
Ignoring wrong pointing object 648 0 (offset 0)
Ignoring wrong pointing object 650 0 (offset 0)
Ignoring wrong pointing object 651 0 (offset 0)
Ignoring wrong pointing object 655 0 (offset

Processing page 2/228 ...


Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 90 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 208 0 (offset 0)
Ignoring wrong pointing object 210 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset 0)
Ignoring wrong pointing object 265 0 (offset 0)
Ignoring wrong pointing object 426 0 (offset 0)
Ignoring wrong pointing object 622 0 (offset 0)
Ignoring wrong pointing object 623 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 626 0 (offset 0)
Ignoring wrong pointing object 627 0 (offset 0)
Ignoring wrong pointing object 633 0 (offset 0)
Ignoring wrong pointing object 647 0 (offset 0)
Ignoring wrong pointing object 648 0 (offset 0)
Ignoring wrong pointing object 650 0 (offset 0)
Ignoring wrong pointing object 651 0 (offset 0)
Ignoring wrong pointing object 655 0 (offset

Processing page 3/228 ...


...

Processing page 227/228 ...


Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 90 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 208 0 (offset 0)
Ignoring wrong pointing object 210 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset 0)
Ignoring wrong pointing object 265 0 (offset 0)
Ignoring wrong pointing object 426 0 (offset 0)
Ignoring wrong pointing object 622 0 (offset 0)
Ignoring wrong pointing object 623 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 626 0 (offset 0)
Ignoring wrong pointing object 627 0 (offset 0)
Ignoring wrong pointing object 633 0 (offset 0)
Ignoring wrong pointing object 647 0 (offset 0)
Ignoring wrong pointing object 648 0 (offset 0)
Ignoring wrong pointing object 650 0 (offset 0)
Ignoring wrong pointing object 651 0 (offset 0)
Ignoring wrong pointing object 655 0 (offset

Processing page 228/228 ...


Ignoring wrong pointing object 623 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 626 0 (offset 0)
Ignoring wrong pointing object 627 0 (offset 0)
Ignoring wrong pointing object 633 0 (offset 0)
Ignoring wrong pointing object 647 0 (offset 0)
Ignoring wrong pointing object 648 0 (offset 0)
Ignoring wrong pointing object 650 0 (offset 0)
Ignoring wrong pointing object 651 0 (offset 0)
Ignoring wrong pointing object 655 0 (offset 0)
Ignoring wrong pointing object 656 0 (offset 0)
Ignoring wrong pointing object 658 0 (offset 0)
Ignoring wrong pointing object 659 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 90 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 208 0 (offset 0)
Ignoring wrong pointing object 210 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset

(228,
 dict_keys(['doc_name', 'page_number', 'has_text_layer', 'image_count', 'text', 'tables']))

In [13]:
all_chunks: List[Dict[str, Any]] = []

with open(JSONL_PATH, "w", encoding="utf-8") as fjsonl:
    for payload in payloads:
        # lưu markdown theo trang nếu muốn
        if SAVE_PAGE_MARKDOWN:
            md_path = MARKDOWN_DIR / f"page_{payload['page_number']:03d}.md"
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(f"# {payload['doc_name']} — Trang {payload['page_number']}\n\n")

                if payload["text"]:
                    f.write("## Text\n\n")
                    f.write(payload["text"] + "\n\n")

                if payload["tables"]:
                    for t in payload["tables"]:
                        f.write(f"## Table {t['table_index']}\n\n")
                        f.write(t["markdown"] + "\n\n")

        # tạo chunks
        page_chunks = build_chunks_from_page(payload)
        all_chunks.extend(page_chunks)

        # ghi JSONL từng chunk
        for ch in page_chunks:
            fjsonl.write(json.dumps(ch, ensure_ascii=False) + "\n")

# xuất tất cả bảng sang Excel
table_records = []
for payload in payloads:
    for t in payload["tables"]:
        table_records.append({
            "doc_name": payload["doc_name"],
            "page_number": payload["page_number"],
            "table_index": t["table_index"],
            "rows": t["rows"],
            "cols": t["cols"],
            "markdown": t["markdown"],
            "csv": t["csv"],
        })

if table_records:
    with pd.ExcelWriter(TABLES_XLSX_PATH, engine="openpyxl") as writer:
        for rec in table_records:
            # lưu mỗi bảng một sheet
            df = pd.read_csv(pd.io.common.StringIO(rec["csv"]))
            sheet_name = f"p{rec['page_number']}_t{rec['table_index']}"
            sheet_name = sheet_name[:31]  # giới hạn của Excel
            df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"JSONL saved to: {JSONL_PATH}")
print(f"Markdown pages saved to: {MARKDOWN_DIR}")
print(f"Tables Excel saved to: {TABLES_XLSX_PATH}")
print(f"Total chunks: {len(all_chunks)}")

JSONL saved to: ..\data\processed\pdf_output\processed_chunks.jsonl
Markdown pages saved to: ..\data\processed\pdf_output\markdown_pages
Tables Excel saved to: ..\data\processed\pdf_output\extracted_tables.xlsx
Total chunks: 452


In [14]:
# xem vài chunk đầu
for i, ch in enumerate(all_chunks[:5], start=1):
    print("=" * 100)
    print(f"Chunk {i}: {ch['chunk_type']} | page {ch['page_number']} | index {ch['chunk_index']}")
    print(ch["content"][:1500])
    print()

Chunk 1: text | page 1 | index 1
BỘ Y TẾ

HƢỚNG DẪN
CHẨN ĐOÁN VÀ XỬ TRÍ NGỘ ĐỘC
(Ban hành kèm theo Quyết định số 3610/QĐ-BYT ngày 31/8/2015
của Bộ trưởng Bộ Y tế)

Hà Nội, 2015

Chunk 2: text | page 3 | index 1
Chủ biên
PGS.TS. Nguyễn Thị Xuyên

Đồng Chủ biên:
PGS.TS. Nguyễn Quốc Anh
PGS.TS. Phạm Duệ
PGS.TS. Lƣơng Ngọc Khuê

Ban biên soạn
PGS.TS. Phạm Duệ
PGS.TS. Bế Hồng Thu
PGS.TS. Hoàng Công Minh
TS. Nguyễn Kim Sơn
TS. Trần Quý Tƣờng
TS. Hà Trần Hƣng
TS. Lê Đức Nhân
BSCK II. Đặng Thị Xuân
Ths. Nguyễn Trung Nguyên
Ths. Nguyễn Tiến Dũng
Ths. Lê Khắc Quyến
Ths. Lê Quang Thuận
Ths. Nguyễn Anh Tuấn
Ths. Nguyễn Đàm Chính

Thƣ kí
Ths. Nguyễn Đức Tiến
Ths. Nguyễn Trung Nguyên
Ths. Ngô Thị Bích Hà
Ths. Trƣơng Lê Vân Ngọc
Ths. Lê Văn Trụ

Chunk 3: text | page 4 | index 1
MỤC LỤC

CÁC CHỮ VIẾT TẮT
CHƢƠNG I: CÁC BIỆN PHÁP CHUNG ..................................................................... 1

1. CHẨN ĐOÁN VÀ XỬ TRÍ CHUNG VỚI NGỘ ĐỘC CẤP ....................................... 1

CHƢƠNG 2:

In [17]:
def load_chunks_from_jsonl(jsonl_path: Path) -> List[Dict[str, Any]]:
    chunks = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    return chunks


def search_chunks(chunks: List[Dict[str, Any]], query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Tìm kiếm cực đơn giản bằng keyword match.
    Dùng tạm để debug, sau đó thay bằng embedding/vector DB.
    """
    q = query.lower().strip()
    scored = []

    for ch in chunks:
        text = ch["content"].lower()
        score = sum(1 for token in q.split() if token in text)
        if score > 0:
            scored.append((score, ch))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [ch for _, ch in scored[:top_k]]


# ví dụ
query = "bảng điểm atropin"
hits = search_chunks(all_chunks, query, top_k=5)

for h in hits:
    print("=" * 100)
    print(h["chunk_type"], h["page_number"], h["chunk_index"])
    print(h["content"][:1200])

text 40 1
atropin, sau đó tạm ngừng cho đến khi hết dấu ngấm. Căn cứ thời gian và liều
đã dùng mà tính ra liều atropine cần duy trì.
- Sử dụng bảng điểm atropin để điều chỉnh liều atropin nguyên tắc dùng liều
thấp nhất để đạt đƣợc dấu thấm. Ngừng atropin khi liều giảm tới 2mg/24 giờ.
- Xử trí khi quá liều: tạm ngừng atropin, theo dõi sát, nếu kích thích vật vã
nhiều có thể cho diazepam (Seduxen tiêm TM); đến khi hết dấu ngấm atropin
thì cho lại atropin với liều thấp hơn liều trƣớc đó.
Bảng 6.1: Bảng điểm atropin

Triệu chứng Ngấm atropin Điểm Quá liều atropin Điểm

1. Da Hồng, ấm 1 Nóng, đỏ 2

2. Đồng tử 3 – 5 mm 1 > 5mm 2

3. Mạch 70 -100lần/phút 1 > 100 lần/phút 2

3. Hô hấp Không tăng tiết, 1 Đờm khô quánh hoặc 2

không co thắt còn không có đờm

đờm dãi lỏng

5. Tinh thần Bình thƣờng 0 Kích thích vật vã, 2

sảng hoặc li bì do

atropin.

6. Bụng Mềm bình thƣờng 0 Chƣớng, gõ trong 2

7.Cầu BQ Không có 0 Căng 2

Cộng 61 62

Điểm A = 61+ 62:
- Điểm A < 4 thiếu atropin phải tăng liều
- Đ